# EMR on EKS with RAPIDS Example

This notebook demonstrates how to submit Spark jobs with RAPIDS acceleration to EMR on EKS from JupyterHub.


In [ ]:
import boto3
import json
import os
import time
from datetime import datetime

# Initialize EMR Containers client
emr_client = boto3.client('emr-containers', region_name=os.environ['AWS_DEFAULT_REGION'])
s3_client = boto3.client('s3')

# Configuration from environment variables
VIRTUAL_CLUSTER_ID = os.environ['EMR_VIRTUAL_CLUSTER_ID']
EXECUTION_ROLE_ARN = os.environ['EMR_EXECUTION_ROLE_ARN']
S3_BUCKET = os.environ['S3_BUCKET']

print(f"Virtual Cluster ID: {VIRTUAL_CLUSTER_ID}")
print(f"S3 Bucket: {S3_BUCKET}")

## Submit RAPIDS-enabled Spark Job


In [ ]:
# Job configuration for RAPIDS-enabled Spark job
job_config = {
    "name": f"rapids-fraud-detection-{int(time.time())}",
    "virtualClusterId": VIRTUAL_CLUSTER_ID,
    "executionRoleArn": EXECUTION_ROLE_ARN,
    "releaseLabel": "emr-6.15.0-latest",
    "jobDriver": {
        "sparkSubmitJobDriver": {
            "entryPoint": f"s3://{S3_BUCKET}/scripts/fraud_detection_feature_engineering.py",
            "entryPointArguments": [
                "--input-path", f"s3://{S3_BUCKET}/data/input/",
                "--output-path", f"s3://{S3_BUCKET}/data/output/",
                "--enable-rapids", "true"
            ],
            "sparkSubmitParameters": " ".join([
                "--conf spark.executor.instances=4",
                "--conf spark.executor.memory=30G",
                "--conf spark.executor.cores=4",
                "--conf spark.executor.resource.gpu.amount=1",
                "--conf spark.task.resource.gpu.amount=0.25",
                "--conf spark.plugins=com.nvidia.spark.SQLPlugin",
                "--conf spark.rapids.sql.enabled=true",
                "--conf spark.rapids.sql.incompatibleOps.enabled=true",
                "--conf spark.sql.adaptive.enabled=false",
                "--conf spark.sql.adaptive.coalescePartitions.enabled=false",
                "--conf spark.kubernetes.executor.podTemplateFile=s3://" + S3_BUCKET + "/pod-templates/executor-pod-template.yaml",
                "--conf spark.kubernetes.driver.podTemplateFile=s3://" + S3_BUCKET + "/pod-templates/driver-pod-template.yaml"
            ])
        }
    },
    "configurationOverrides": {
        "applicationConfiguration": [
            {
                "classification": "spark-defaults",
                "properties": {
                    "spark.kubernetes.container.image": f"{boto3.Session().region_name}.dkr.ecr.{boto3.Session().region_name}.amazonaws.com/spark-rapids:latest",
                    "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
                    "spark.kubernetes.executor.label.type": "spark-executor-gpu",
                    "spark.kubernetes.driver.label.type": "spark-driver"
                }
            }
        ],
        "monitoringConfiguration": {
            "cloudWatchMonitoringConfiguration": {
                "logGroupName": f"/aws/emr-containers/{VIRTUAL_CLUSTER_ID}",
                "logStreamNamePrefix": "rapids-fraud-detection"
            },
            "s3MonitoringConfiguration": {
                "logUri": f"s3://{S3_BUCKET}/logs/"
            }
        }
    }
}

print("Job Configuration:")
print(json.dumps(job_config, indent=2))

In [ ]:
# Submit the job
response = emr_client.start_job_run(**job_config)
job_run_id = response['id']

print(f"Job submitted successfully!")
print(f"Job Run ID: {job_run_id}")
print(f"Job ARN: {response['arn']}")

## Monitor Job Progress


In [ ]:
def monitor_job(job_run_id, max_wait_time=1800):  # 30 minutes max
    """Monitor EMR job progress"""
    start_time = time.time()
    
    while time.time() - start_time < max_wait_time:
        response = emr_client.describe_job_run(
            virtualClusterId=VIRTUAL_CLUSTER_ID,
            id=job_run_id
        )
        
        job_run = response['jobRun']
        state = job_run['state']
        
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Job State: {state}")
        
        if state in ['COMPLETED', 'FAILED', 'CANCELLED']:
            if state == 'COMPLETED':
                print("✅ Job completed successfully!")
            else:
                print(f"❌ Job {state.lower()}")
                if 'stateDetails' in job_run:
                    print(f"Details: {job_run['stateDetails']}")
            break
        
        time.sleep(30)  # Wait 30 seconds before checking again
    
    return response

# Monitor the job
final_status = monitor_job(job_run_id)

## View Job Logs


In [ ]:
# Get CloudWatch logs
logs_client = boto3.client('logs', region_name=os.environ['AWS_DEFAULT_REGION'])

log_group_name = f"/aws/emr-containers/{VIRTUAL_CLUSTER_ID}"
log_stream_prefix = f"rapids-fraud-detection/{job_run_id}"

try:
    # List log streams
    streams_response = logs_client.describe_log_streams(
        logGroupName=log_group_name,
        logStreamNamePrefix=log_stream_prefix,
        orderBy='LastEventTime',
        descending=True,
        limit=5
    )
    
    print("Available log streams:")
    for stream in streams_response['logStreams']:
        print(f"- {stream['logStreamName']}")
        
        # Get recent log events
        events_response = logs_client.get_log_events(
            logGroupName=log_group_name,
            logStreamName=stream['logStreamName'],
            limit=10,
            startFromHead=False
        )
        
        print(f"\nRecent logs from {stream['logStreamName']}:")
        for event in events_response['events'][-5:]:  # Show last 5 events
            timestamp = datetime.fromtimestamp(event['timestamp'] / 1000)
            print(f"[{timestamp}] {event['message']}")
        print("-" * 80)
        
except Exception as e:
    print(f"Error retrieving logs: {e}")
    print("Logs may not be available yet or log group may not exist.")

## Check Output Data


In [ ]:
# List output files in S3
output_prefix = "data/output/"

try:
    response = s3_client.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix=output_prefix
    )
    
    if 'Contents' in response:
        print(f"Output files in s3://{S3_BUCKET}/{output_prefix}:")
        for obj in response['Contents']:
            print(f"- {obj['Key']} ({obj['Size']} bytes, {obj['LastModified']})")
    else:
        print("No output files found yet.")
        
except Exception as e:
    print(f"Error listing S3 objects: {e}")

## Cleanup (Optional)


In [ ]:
# Cancel job if needed (uncomment to use)
# emr_client.cancel_job_run(
#     virtualClusterId=VIRTUAL_CLUSTER_ID,
#     id=job_run_id
# )
# print(f"Job {job_run_id} cancelled")